[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.4_multi_model_gateway/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.4_multi_model_gateway/lab.ipynb)

# 10.4 Lab: Multi-Model Gateway Simulator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.4_multi_model_gateway/lab.ipynb) [![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.cloud/github/harshuljain13/llm-inference-at-scale/blob/master/content/11_system_designs/10.4_multi_model_gateway/lab.ipynb)

Simulate a multi-model gateway routing 10K requests across a heterogeneous fleet. Measure cost savings, quality trade-offs, and routing accuracy.


In [ ]:
# Install dependencies via subprocess for Colab/Molab compatibility
import subprocess
import sys
# numpy: numerical arrays and random generation
# matplotlib: visualization of routing decisions and cost
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy", "matplotlib"])


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# === PARAMETERS (modify these and re-run the notebook) ===
# Number of inference requests to simulate in this experiment
NUM_REQUESTS = 10000
# Random seed for reproducible results
SEED = 42

# Model catalog defines the heterogeneous fleet
# Each model has: quality scores per complexity tier, cost per 1K tokens, latency in ms
# Quality is indexed 0-4 mapping to complexity tiers (trivial -> extreme)
MODEL_CATALOG = {
    # Largest model: highest quality but most expensive and slowest
    "405B": {"quality": [9.5, 9.4, 9.2, 9.0, 8.5], "cost": 0.015, "latency": 1200},
    # General-purpose workhorse: good quality/cost balance
    "70B":  {"quality": [9.3, 8.8, 7.8, 6.5, 5.2], "cost": 0.004, "latency": 350},
    # Fast and cheap: excellent for simple tasks (9.0 quality on trivial!)
    "8B":   {"quality": [9.0, 7.5, 6.1, 4.8, 3.5], "cost": 0.0008, "latency": 100},
    # Classifier/router model: minimal cost, used for complexity estimation
    "1.5B": {"quality": [7.5, 5.0, 3.8, 2.5, 1.8], "cost": 0.0001, "latency": 20},
}

# How real traffic distributes across complexity tiers
# 65% of requests are simple (tier 0-1), only 5% are extreme (tier 4)
COMPLEXITY_DIST = [0.30, 0.35, 0.20, 0.10, 0.05]


In [ ]:
def route_request_gateway(complexity_tier, min_quality=6.0, max_latency=5000):
    """Route one request to the cheapest model meeting quality+latency constraints.
    
    The gateway solves: maximize(quality/cost) subject to quality >= floor, latency <= ceiling.
    Returns: (model_name, cost, quality_delivered)
    """
    # Collect all models that meet hard constraints for this request
    candidates = []
    for name, spec in MODEL_CATALOG.items():
        # Look up this model's expected quality for the given complexity tier
        model_quality = spec["quality"][complexity_tier]
        # Reject models below quality floor (would deliver unacceptable output)
        if model_quality < min_quality:
            continue
        # Reject models too slow for this team's latency SLO
        if spec["latency"] > max_latency:
            continue
        # Calculate cost-efficiency: quality delivered per dollar spent
        efficiency = model_quality / spec["cost"]
        # Store candidate with its metrics for ranking
        candidates.append((name, spec["cost"], model_quality, efficiency))
    
    # If no model meets constraints, fall back to cheapest (degrade gracefully)
    if not candidates:
        return "1.5B", MODEL_CATALOG["1.5B"]["cost"], MODEL_CATALOG["1.5B"]["quality"][complexity_tier]
    
    # Rank by cost-efficiency: best quality per dollar at the top
    candidates.sort(key=lambda x: x[3], reverse=True)
    # Select the most cost-efficient model that meets all constraints
    best = candidates[0]
    return best[0], best[1], best[2]

def route_request_naive(complexity_tier):
    """Naive strategy: send everything to 405B regardless of complexity.
    
    This is what orgs do without a gateway -- overspend on simple requests.
    """
    # Always use the largest, most expensive model
    spec = MODEL_CATALOG["405B"]
    # Return: model name, cost (always highest), quality (always max)
    return "405B", spec["cost"], spec["quality"][complexity_tier]


In [ ]:
# === SIMULATION: Compare gateway routing vs naive all-405B strategy ===
# Initialize random number generator with fixed seed
rng = np.random.default_rng(SEED)

# Sample complexity tiers for each request using the traffic distribution
# This simulates real-world traffic where most requests are simple
complexities = rng.choice(len(COMPLEXITY_DIST), size=NUM_REQUESTS, p=COMPLEXITY_DIST)

# Accumulate per-request metrics for both strategies
gateway_costs = []      # cost incurred per request with intelligent routing
gateway_qualities = []  # quality score delivered per request
gateway_models = []     # which model handled each request
naive_costs = []        # cost incurred per request with naive routing
naive_qualities = []    # quality delivered (always max since always 405B)

# Process each request through both routing strategies
for comp in complexities:
    # Gateway: route based on complexity, quality floor, latency ceiling
    g_model, g_cost, g_quality = route_request_gateway(comp)
    gateway_costs.append(g_cost)
    gateway_qualities.append(g_quality)
    gateway_models.append(g_model)
    
    # Naive: always use 405B, ignoring complexity (the baseline to beat)
    _, n_cost, n_quality = route_request_naive(comp)
    naive_costs.append(n_cost)
    naive_qualities.append(n_quality)

# Convert lists to numpy arrays for vectorized statistics
gateway_costs = np.array(gateway_costs)
naive_costs = np.array(naive_costs)
gateway_qualities = np.array(gateway_qualities)
naive_qualities = np.array(naive_qualities)

# Report key comparison metrics
print(f"=== Routing Simulation ({NUM_REQUESTS:,} requests) ===")
print(f"\nGateway total cost:  ${gateway_costs.sum():.2f}")
print(f"Naive total cost:    ${naive_costs.sum():.2f}")
# Cost savings percentage: how much the gateway saves vs naive approach
print(f"Cost savings:        {(1 - gateway_costs.sum()/naive_costs.sum())*100:.1f}%")
print(f"\nGateway avg quality: {gateway_qualities.mean():.2f}/10")
print(f"Naive avg quality:   {naive_qualities.mean():.2f}/10")
# Quality delta: how much quality we sacrifice for the cost savings
print(f"Quality delta:       {(naive_qualities.mean() - gateway_qualities.mean()):.2f} points")


In [ ]:
# === VISUALIZATION 1: Model distribution and cost comparison ===
# Count how many requests the gateway routed to each model
model_names = list(MODEL_CATALOG.keys())
model_counts = [gateway_models.count(m) for m in model_names]
# Calculate traffic percentage per model tier
model_pcts = [c / NUM_REQUESTS * 100 for c in model_counts]

# Pastel colors matching the module's mermaid diagram palette
colors = ["#f3e8ff", "#dbeafe", "#dcfce7", "#f3f4f6"]

# Create side-by-side subplots for distribution and cost
fig_c4, axes_c4 = plt.subplots(1, 2, figsize=(12, 5))

# Left panel: pie chart showing where traffic actually goes
axes_c4[0].pie(model_counts, labels=model_names, colors=colors, autopct="%1.1f%%",
            startangle=90, textprops={"fontsize": 11})
axes_c4[0].set_title("Request Distribution by Model", fontsize=13, fontweight="bold")

# Right panel: bar chart comparing total cost between strategies
categories = ["Gateway\n(intelligent)", "Naive\n(all 405B)"]
costs = [gateway_costs.sum(), naive_costs.sum()]
# Green for savings, red for waste
bar_colors = ["#dcfce7", "#ffe4e6"]
bars = axes_c4[1].bar(categories, costs, color=bar_colors, edgecolor="#000", linewidth=1.2)
axes_c4[1].set_ylabel("Total Cost ($)", fontsize=11)
axes_c4[1].set_title("Cost: Gateway vs Naive Routing", fontsize=13, fontweight="bold")
# Add dollar labels on top of each bar for quick reading
for bar, cost in zip(bars, costs):
    axes_c4[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"${cost:.2f}", ha="center", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("gateway_routing_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: gateway_routing_analysis.png")


In [ ]:
# === VISUALIZATION 2: Quality vs cost scatter by complexity tier ===
fig_c5, ax_c5 = plt.subplots(figsize=(10, 6))

# Human-readable labels for the 5 complexity tiers
tier_labels = ["Trivial (1-2)", "Simple (3-4)", "Moderate (5-6)", "Hard (7-8)", "Extreme (9-10)"]
# Distinct colors per tier for visual separation
tier_colors = ["#dcfce7", "#dbeafe", "#fef3c7", "#ffedd5", "#ffe4e6"]

# Plot each tier as a separate scatter series (enables legend)
for tier in range(5):
    # Boolean mask selecting requests with this complexity tier
    mask = complexities == tier
    # Scatter: x=cost incurred, y=quality delivered, colored by tier
    ax_c5.scatter(gateway_costs[mask], gateway_qualities[mask],
               c=tier_colors[tier], edgecolors="#000", linewidth=0.5,
               s=40, alpha=0.7, label=tier_labels[tier])

# Axis labels describing what each dimension represents
ax_c5.set_xlabel("Cost per Request ($)", fontsize=11)
ax_c5.set_ylabel("Quality Score Delivered", fontsize=11)
ax_c5.set_title("Gateway Routing: Quality vs Cost by Complexity Tier", fontsize=13, fontweight="bold")
# Legend shows which color corresponds to which complexity level
ax_c5.legend(title="Complexity Tier", loc="lower right")
ax_c5.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("quality_vs_cost_scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: quality_vs_cost_scatter.png")


In [ ]:
# === EXPERIMENT 3: Budget enforcement simulation over 30 days ===
# Simulates what happens when a team hits their monthly budget limit
TEAM_BUDGET_USD = 50.0   # monthly budget cap for one team
REQUESTS_PER_DAY = 1000  # how many requests this team generates daily

# Track costs over the month
daily_costs = []         # cost incurred each day
cumulative_costs = []    # running total across all days
budget_exceeded_day = None  # which day the budget runs out

running_total = 0.0
for day in range(30):
    # Generate one day of traffic with random complexity distribution
    daily_complexities = rng.choice(len(COMPLEXITY_DIST), size=REQUESTS_PER_DAY, p=COMPLEXITY_DIST)
    day_cost = 0.0
    
    for comp in daily_complexities:
        # Check if team has budget remaining before routing
        if running_total + day_cost >= TEAM_BUDGET_USD:
            # Budget exhausted: gateway forces fallback to cheapest model (1.5B)
            # Quality degrades but service continues (graceful degradation)
            day_cost += MODEL_CATALOG["1.5B"]["cost"]
        else:
            # Budget available: use intelligent routing as normal
            _, cost, _ = route_request_gateway(comp)
            day_cost += cost
    
    # Record this day's cost and update running total
    daily_costs.append(day_cost)
    running_total += day_cost
    cumulative_costs.append(running_total)
    
    # Track the first day budget was exceeded (for annotation)
    if running_total >= TEAM_BUDGET_USD and budget_exceeded_day is None:
        budget_exceeded_day = day + 1

# Plot cumulative cost with budget line
fig_c6, ax_c6 = plt.subplots(figsize=(10, 5))
# Blue line: actual cost accumulation over time
ax_c6.plot(range(1, 31), cumulative_costs, color="#2563eb", linewidth=2, label="Cumulative Cost")
# Red dashed line: budget ceiling (hard limit)
ax_c6.axhline(y=TEAM_BUDGET_USD, color="#991b1b", linestyle="--", linewidth=1.5,
           label=f"Budget Limit (${TEAM_BUDGET_USD})")
# Annotate the day budget was hit (if within 30 days)
if budget_exceeded_day:
    ax_c6.axvline(x=budget_exceeded_day, color="#991b1b", linestyle=":", alpha=0.7)
    ax_c6.annotate(f"Budget hit: Day {budget_exceeded_day}",
                xy=(budget_exceeded_day, TEAM_BUDGET_USD),
                xytext=(budget_exceeded_day + 2, TEAM_BUDGET_USD * 0.8),
                arrowprops=dict(arrowstyle="->"), fontsize=10)

ax_c6.set_xlabel("Day of Month", fontsize=11)
ax_c6.set_ylabel("Cumulative Cost ($)", fontsize=11)
ax_c6.set_title("Team Budget Enforcement Over 30 Days", fontsize=13, fontweight="bold")
ax_c6.legend(fontsize=10)
ax_c6.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("budget_enforcement.png", dpi=150, bbox_inches="tight")
plt.show()
# Report when team hit the budget wall
print(f"Budget exceeded on day: {budget_exceeded_day or 'Never (within 30 days)'}")
